# Study 05: Train From Scratch (50 Epochs)\n**Goal:** Train without ImageNet pretraining to measure the impact of transfer learning.

## 1. Setup & GPU Check

In [ ]:
import sys, json, os
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import torch
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')
print(f'Torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## 2. Configuration

In [ ]:
EPOCHS = 5; BATCH_SIZE = 4; LR = 1e-3; SEED = 42
OUTPUT_DIR = Path('../models/s9_pilot')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Output: {OUTPUT_DIR}\nEpochs: {EPOCHS}, Batch: {BATCH_SIZE}, LR: {LR}, Seed: {SEED}')
from src.utils import set_seed; set_seed(SEED)

## 5. Training (50 epochs from scratch — ~5 hrs)

In [ ]:
print('Starting 50-epoch fine-tune...')
trainer.train(warmup_epochs=5)
best_dice = trainer.best_val_dice
print(f'Best val dice: {best_dice:.4f}')

## 4. Model & Trainer Setup

In [ ]:
from src.models import create_model, count_params
from src.trainer import Trainer
from src.config import PHASE4_RESEARCH_CONFIG

model = create_model('mobilenetv2_unet', in_channels=1, out_channels=1, pretrained=True).to(DEVICE)
print(f'Parameters: {count_params(model):,}')

train_config = dict(PHASE4_RESEARCH_CONFIG)
train_config['use_focal'] = True
trainer = Trainer(model=model, train_loader=train_loader, val_loader=val_loader,
    config=train_config, learning_rate=LR, num_epochs=EPOCHS,
    mixed_precision=True, output_dir=str(OUTPUT_DIR))
print('Trainer initialized with FocalLoss')

## 5. Training (5 epochs)

In [ ]:
print('Starting training...')
trainer.train(warmup_epochs=0)
best_dice = trainer.best_val_dice
print(f'Best val dice: {best_dice:.4f}')
print(f'Best at epoch: {1+trainer.history["val_dice"].index(max(trainer.history["val_dice"]))}')

## 6. Training Curves

In [ ]:
hist = trainer.history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(hist['val_dice'], 'b-o', label='Train')
axes[0].plot(hist['val_dice'], 'r-o', label='Validation')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Dice'); axes[0].set_title('Dice Coefficient')
axes[0].legend(); axes[0].grid(True)
axes[1].plot(hist['train_loss'], 'b-o', label='Train')
axes[1].plot(hist['val_iou'], 'r-o', label='Validation')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss'); axes[1].set_title('Loss')
axes[1].legend(); axes[1].grid(True)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_curve.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Test Set Evaluation

In [ ]:
test_metrics = trainer.evaluate(test_loader)
print(f'Test Dice: {test_metrics["dice"]:.4f}')
print(f'Test IoU:  {test_metrics["iou"]:.4f}')
print(f'Test HD95: {test_metrics.get("hd95", "N/A")}')

with open(OUTPUT_DIR / 'history.json', 'w') as f:
    json.dump(trainer.history, f, indent=2)
with open(OUTPUT_DIR / 'test_metrics.json', 'w') as f:
    json.dump(test_metrics, f, indent=2)
print('Results saved')

## 8. Summary

In [ ]:
print('='*60)
print('PILOT TRAINING SUMMARY')
print('='*60)
print(f'Epochs: {EPOCHS}')
print(f'Loss: FocalLoss (alpha=0.25, gamma=2.0)')
print(f'Best val dice: {best_dice:.4f}')
print(f'Test dice: {test_metrics["dice"]:.4f}')
print(f'Test IoU: {test_metrics["iou"]:.4f}')
print(f'\nNext: Run Study 05 (scratch training) if fine-tune looks good')